# Dead Time Analysis — Non-Paralyzable Detector
**Method:** Fit the shifted exponential $p(\Delta t) = \lambda\, e^{-\lambda(\Delta t - \tau)}$ for $\Delta t \geq \tau$ to 6 inter-arrival time distributions.  
$\tau$ is **shared** across all runs (detector property); $\lambda_i$ is the **true rate** for run $i$.  
Individual source activities follow as $r_i = \lambda_i - \lambda_{i-1}$.

## The key property of the non-paralyzable model

For a non paralyzable detector with true rate $\lambda$, the inter-arrival time PDF between recorded events is:

$p(t) = \lambda \mathrm{e}^{-\lambda(t - \tau)}, \,\, t \geq \tau$

The crucial point: the exponential slope gives you the true rate $\lambda$ directly -- not the observed (reduced) rate. The date time only shifts the distribution; it does not change the decay constant. This is the most powerful tool.


## Cell 1 — Imports

In [1]:
import ROOT
import pandas as pd
import numpy as np
from scipy.optimize import minimize
import warnings
warnings.filterwarnings('ignore')

ROOT.gROOT.SetBatch(False)
ROOT.gStyle.SetOptStat(0)
ROOT.gStyle.SetOptFit(111)   # show fit params on plot
ROOT.gStyle.SetPadGridX(True)
ROOT.gStyle.SetPadGridY(True)
print('ROOT version:', ROOT.gROOT.GetVersion())

import ipynbname
import sys
from pathlib import Path
project_root = str(ipynbname.path().parent.parent.parent)
sys.path.append(project_root)
project_root=Path(project_root)

ROOT version: 6.36.06


## Cell 2 — Load data
**Adjust** `FILES` to match your actual filenames.  
If all runs are in **one CSV** with a `run` column, replace the loading block with:
```python
df_all = pd.read_csv('all_runs.csv')
datasets = [df_all[df_all['run']==i]['time_s'].values for i in range(1, 7)]
```

In [2]:
data_path = project_root/'data/260527'
FILES = [list((data_path).glob(f'time_{i}*.csv'))[0] for i in [29,30,31,32,34,35]]
FILES

[PosixPath('/home/oscar/Projects/SIPHRA_Testing/data/260527/time_29_siphraAonly_Timing_Cs-137_1source.csv'),
 PosixPath('/home/oscar/Projects/SIPHRA_Testing/data/260527/time_30_siphraAonly_Timing_Cs-137_2sources.csv'),
 PosixPath('/home/oscar/Projects/SIPHRA_Testing/data/260527/time_31_siphraAonly_Timing_Cs-137_3sources.csv'),
 PosixPath('/home/oscar/Projects/SIPHRA_Testing/data/260527/time_32_siphraAonly_Timing_Cs-137_4sources.csv'),
 PosixPath('/home/oscar/Projects/SIPHRA_Testing/data/260527/time_34_siphraAonly_Timing_Cs-137_5sources.csv'),
 PosixPath('/home/oscar/Projects/SIPHRA_Testing/data/260527/time_35_siphraAonly_Timing_Cs-137_6sources.csv')]

In [3]:
datasets = []
for i, fname in enumerate(FILES):
    df = pd.read_csv(fname)
    dt = df['time_s'].values.astype(float)
    dt = dt[dt > 0]          # drop any zeroes / artefacts
    datasets.append(dt)

print(f"{'Run':>4}  {'N events':>10}  {'min Δt (s)':>14}  {'mean Δt (s)':>14}  {'1/mean (Hz)':>14}")
print('-' * 62)
for i, dt in enumerate(datasets):
    print(f"{i+1:>4}  {len(dt):>10,}  {dt.min():>14.4e}  {dt.mean():>14.4e}  {1/dt.mean():>14.2f}")

 Run    N events      min Δt (s)     mean Δt (s)     1/mean (Hz)
--------------------------------------------------------------
   1     500,000      1.2000e-05      4.7683e-04         2097.20
   2     500,000      1.2000e-05      3.7966e-04         2633.97
   3     500,000      1.2000e-05      3.0813e-04         3245.43
   4     500,000      1.1000e-05      2.4317e-04         4112.39
   5     500,000      1.2000e-05      1.7851e-04         5601.92
   6     500,000      1.1000e-05      1.7030e-04         5871.83


## Cell 3 — First look: choose histogram binning
Key choices:
- **Lower edge = 0** so the dead-time onset is visible.
- **Bin width** should be small enough to resolve the onset at $\tau$ but not so small that bins are empty in the tail.
- A rule of thumb: aim for ≥ 10 counts/bin in the bulk of the distribution.

In [ ]:
# Use the run with the HIGHEST rate to set the time axis
# (it has the shortest mean Δt and will drive the τ estimate)
means = np.array([dt.mean() for dt in datasets])

# Upper edge: 6 × mean of the LOWEST-rate run (run 1)
T_MAX  = 6.0 * means[0]

# Bin width: mean of the HIGHEST-rate run / 40  (empirical, adjust if needed)
BIN_W  = means[-1] / 40.0
N_BINS = int(T_MAX / BIN_W)

print(f'T_MAX  = {T_MAX:.4e} s')
print(f'BIN_W  = {BIN_W:.4e} s')
print(f'N_BINS = {N_BINS}')

## Cell 4 — Build ROOT histograms

In [ ]:
COLORS = [ROOT.kBlue+1, ROOT.kRed+1, ROOT.kGreen+2,
          ROOT.kMagenta+1, ROOT.kCyan+2, ROOT.kOrange+1]

hists = []
for i, dt in enumerate(datasets):
    h = ROOT.TH1D(f'h{i+1}', f'Run {i+1};#Delta t (s);Counts / bin',
                  N_BINS, 0.0, T_MAX)
    h.SetDirectory(0)      # keep alive outside any TFile
    h.SetLineColor(COLORS[i])
    h.SetLineWidth(2)
    for t in dt:
        if 0 < t <= T_MAX:
            h.Fill(t)
    hists.append(h)
    print(f'Run {i+1}: {h.GetEntries():.0f} entries filled')

## Cell 5 — Visual inspection (log-y)
On a log-y scale a shifted exponential is a straight line starting at $\tau$.  
Look for the **onset** (where counts drop to zero at small $\Delta t$) — that is $\tau$.

In [ ]:
c_inspect = ROOT.TCanvas('c_inspect', 'All runs — log scale', 1400, 900)
c_inspect.Divide(3, 2)
for i, h in enumerate(hists):
    c_inspect.cd(i + 1)
    ROOT.gPad.SetLogy(1)
    h.Draw('HIST')
c_inspect.Draw()

## Cell 6 — Initial τ estimate and preliminary λᵢ
The global minimum of all inter-arrival times is a **lower bound** on $\tau$ (the true value is slightly higher due to finite statistics). We use it as a starting point.

For each run we get a rough $\lambda_i$ from $1 / (\bar{\Delta t}_i - \hat{\tau})$.

In [ ]:
ns    = np.array([len(dt)      for dt in datasets])   # event counts
mins  = np.array([dt.min()     for dt in datasets])   # minimum per run
means = np.array([dt.mean()    for dt in datasets])   # mean per run

# Conservative initial τ: global minimum over all runs
TAU_INIT = mins.min()

# Upper bound for optimisation: no Δt can be smaller than τ
TAU_MAX  = mins.min()   # hard ceiling

lam_init = 1.0 / (means - TAU_INIT)

print(f'Global minimum Δt (τ upper bound) = {TAU_MAX:.6e} s')
print()
print(f"{'Run':>4}  {'λ_init (Hz)':>14}")
for i, lam in enumerate(lam_init):
    print(f'{i+1:>4}  {lam:>14.4f}')

## Cell 7 — Individual ROOT fits (diagnostic)
Fit each histogram independently to $A\,e^{-\lambda(\Delta t - \tau)}$ to get per-run estimates of $\lambda_i$ and $\tau$.  
The spread in the fitted $\tau$ values tells you how consistent the detector model is.

> **Note on fit range:** we start the fit at `FIT_START = TAU_INIT * 1.05` to stay safely above the onset and avoid the partial first bin.

In [ ]:
FIT_START = TAU_INIT * 1.05    # just above onset

ind_lambdas = []
ind_taus    = []
ind_errs_l  = []
ind_errs_t  = []
fit_funcs   = []

c_indiv = ROOT.TCanvas('c_indiv', 'Individual fits', 1400, 900)
c_indiv.Divide(3, 2)

for i, h in enumerate(hists):
    c_indiv.cd(i + 1)
    ROOT.gPad.SetLogy(1)

    f = ROOT.TF1(f'f_ind{i+1}',
                 '[0] * exp( -[1] * (x - [2]) )',
                 FIT_START, T_MAX)
    f.SetParNames('A', '#lambda', '#tau')

    # Initial values
    f.SetParameter(0, h.GetBinContent(h.FindBin(FIT_START * 1.1)))
    f.SetParameter(1, lam_init[i])
    f.SetParameter(2, TAU_INIT)

    # Bounds
    f.SetParLimits(0, 1e-3, 1e12)
    f.SetParLimits(1, 0.0,  1e12)
    f.SetParLimits(2, 0.0,  TAU_MAX)

    h.Draw('HIST')
    # 'L' = log-likelihood fit (better for low-count bins), 'R' = use TF1 range
    h.Fit(f, 'LRQ')   # Q = quiet
    f.SetLineColor(ROOT.kBlack)
    f.SetLineWidth(2)
    f.Draw('SAME')
    fit_funcs.append(f)

    lam = f.GetParameter(1);  el = f.GetParError(1)
    tau = f.GetParameter(2);  et = f.GetParError(2)
    ind_lambdas.append(lam);  ind_errs_l.append(el)
    ind_taus.append(tau);     ind_errs_t.append(et)

c_indiv.Draw()

print(f"{'Run':>4}  {'λ (Hz)':>14}  {'δλ':>10}  {'τ (s)':>14}  {'δτ':>12}")
print('-' * 62)
for i in range(N_RUNS):
    print(f"{i+1:>4}  {ind_lambdas[i]:>14.4f}  {ind_errs_l[i]:>10.4f}"
          f"  {ind_taus[i]:>14.6e}  {ind_errs_t[i]:>12.4e}")

print(f"\nMean τ (individual fits) = {np.mean(ind_taus):.6e} s")
print(f"Std  τ (individual fits) = {np.std(ind_taus):.2e} s")

## Cell 8 — Global fit with shared τ
**Strategy:** minimise a combined binned Poisson negative log-likelihood over all 6 histograms simultaneously, with $\tau$ shared and $\lambda_i$ individual.

For bin $j$ of run $i$ with edges $[a, b]$ the expected count is:
$$\mu_{ij} = n_i \left[e^{-\lambda_i(a-\tau)} - e^{-\lambda_i(b-\tau)}\right] \quad (a \geq \tau)$$
with the obvious boundary treatment for the bin that straddles $\tau$.

The Poisson NLL is $\sum_{ij}\left[\mu_{ij} - O_{ij}\ln\mu_{ij}\right]$.

In [ ]:
# Pre-extract histogram bin edges and counts (do this once, outside the objective)
bin_edges  = []   # shape: (N_RUNS, N_BINS+1)
bin_counts = []   # shape: (N_RUNS, N_BINS)

for h in hists:
    nb = h.GetNbinsX()
    edges  = np.array([h.GetBinLowEdge(b) for b in range(1, nb + 2)])
    counts = np.array([h.GetBinContent(b) for b in range(1, nb + 1)])
    bin_edges.append(edges)
    bin_counts.append(counts)


def expected_counts(lam, tau, edges, n_total):
    """Expected bin counts for a shifted exponential with rate lam, onset tau."""
    a = edges[:-1]   # left  edges, shape (N_BINS,)
    b = edges[1:]    # right edges

    # Bins entirely below tau: zero
    # Bins entirely above tau: standard integral
    # Bin straddling tau: clamp lower edge to tau
    a_eff = np.maximum(a, tau)   # clamp

    # Integral of lam*exp(-lam*(t-tau)) from a_eff to b
    mu = n_total * (np.exp(-lam * (a_eff - tau)) - np.exp(-lam * (b - tau)))
    mu = np.where(b <= tau, 0.0, mu)   # bins entirely below tau
    mu = np.maximum(mu, 1e-300)        # avoid log(0)
    return mu


def global_nll(params):
    """
    Combined Poisson NLL over all 6 histograms.
    params = [tau, lambda_1, lambda_2, ..., lambda_6]
    """
    tau  = params[0]
    lams = params[1:]

    if tau <= 0 or tau >= TAU_MAX:
        return 1e30
    if np.any(lams <= 0):
        return 1e30

    nll = 0.0
    for i in range(N_RUNS):
        mu = expected_counts(lams[i], tau, bin_edges[i], ns[i])
        obs = bin_counts[i]
        # Poisson NLL: sum(mu - obs*log(mu)), ignore constant log(obs!) terms
        nll += np.sum(mu - obs * np.log(mu))

    return nll


# --- Initial parameter vector: [tau, lam1, ..., lam6] ---
p0 = np.concatenate([[np.mean(ind_taus)], ind_lambdas])

# Bounds: tau in (0, TAU_MAX), each lambda > 0
bounds = [(1e-10, TAU_MAX * 0.9999)] + [(1.0, None)] * N_RUNS

result = minimize(global_nll, p0, method='L-BFGS-B', bounds=bounds,
                  options={'ftol': 1e-12, 'gtol': 1e-8, 'maxiter': 5000})

print('Minimisation status:', result.message)
print(f'Final NLL = {result.fun:.4f}')
print()

tau_fit  = result.x[0]
lams_fit = result.x[1:]

print(f'Global τ  = {tau_fit:.6e} s')
print()
print(f"{'Run':>4}  {'λ_global (Hz)':>16}")
for i, lam in enumerate(lams_fit):
    print(f'{i+1:>4}  {lam:>16.4f}')

## Cell 9 — Uncertainty on τ via profile likelihood scan
We scan $\tau$ around the best-fit value and find the interval where $\Delta(-2\ln\mathcal{L}) < 1$ (68% CL, 1σ).
At each $\tau$, $\lambda_i$ are re-optimised (they are analytic: $\hat{\lambda}_i(\tau) = 1/(\bar{\Delta t}_i - \tau)$, which we use as warm-start).

In [ ]:
N_SCAN   = 300
tau_lo   = max(1e-10,     tau_fit * 0.50)
tau_hi   = min(TAU_MAX * 0.9999, tau_fit * 1.50)
tau_scan = np.linspace(tau_lo, tau_hi, N_SCAN)

nll_scan = []
lams_scan = []
for tau_s in tau_scan:
    # Warm-start λᵢ from analytic profile estimate
    lam_warm = 1.0 / np.maximum(means - tau_s, 1e-12)
    p_scan   = np.concatenate([[tau_s], lam_warm])
    bounds_s = [(tau_s - 1e-15, tau_s + 1e-15)] + [(1.0, None)] * N_RUNS  # fix tau
    res_s = minimize(global_nll, p_scan, method='L-BFGS-B', bounds=bounds_s,
                     options={'ftol': 1e-12, 'maxiter': 2000})
    nll_scan.append(res_s.fun)
    lams_scan.append(res_s.x[1:])

nll_scan  = np.array(nll_scan)
delta_nll = 2.0 * (nll_scan - result.fun)   # -2 Δ ln L

# 1σ interval: where delta_nll = 1
# Since this is a boundary-constrained parameter, the interval is one-sided
crossing = tau_scan[delta_nll < 1.0]
tau_1sig_lo = crossing.min()  if len(crossing) else tau_fit
tau_1sig_hi = crossing.max()  if len(crossing) else tau_fit

print(f'τ = {tau_fit:.6e} s   [{tau_1sig_lo:.4e}, {tau_1sig_hi:.4e}]  (1σ profile interval)')

# Plot the profile
c_profile = ROOT.TCanvas('c_profile', 'Profile likelihood for τ', 800, 500)
g_prof = ROOT.TGraph(len(tau_scan),
                     tau_scan.astype('d'),
                     delta_nll.astype('d'))
g_prof.SetTitle(';#tau (s);-2#Delta ln#it{L}')
g_prof.SetLineWidth(2); g_prof.SetLineColor(ROOT.kBlue+1)
g_prof.Draw('AL')
line1 = ROOT.TLine(tau_lo, 1.0, tau_hi, 1.0)
line1.SetLineStyle(2); line1.SetLineColor(ROOT.kRed)
line1.Draw()
c_profile.Draw()

## Cell 10 — Final overlay: data + global fit
Draw each histogram with the best-fit curve using the globally determined $\tau$ and per-run $\lambda_i$.

In [ ]:
c_final = ROOT.TCanvas('c_final', 'Global fit — all runs', 1400, 900)
c_final.Divide(3, 2)

fit_funcs_global = []
for i, h in enumerate(hists):
    c_final.cd(i + 1)
    ROOT.gPad.SetLogy(1)

    f = ROOT.TF1(f'f_global{i+1}',
                 f'[0] * exp( -[1] * (x - {tau_fit:.8e}) ) * (x >= {tau_fit:.8e})',
                 0, T_MAX)
    # Normalisation: match bin content at the first populated bin
    first_bin = h.FindFirstBinAbove(0)
    norm = h.GetBinContent(first_bin)
    f.SetParameter(0, norm * np.exp(lams_fit[i] *
                                     (h.GetBinCenter(first_bin) - tau_fit)))
    f.SetParameter(1, lams_fit[i])
    f.SetLineColor(ROOT.kBlack); f.SetLineWidth(2)

    h.SetTitle(f'Run {i+1}  (#lambda = {lams_fit[i]:.2f} Hz);#Delta t (s);Counts / bin')
    h.Draw('HIST')
    f.Draw('SAME')

    # Add τ marker
    y_max = h.GetMaximum()
    line_tau = ROOT.TLine(tau_fit, h.GetMinimum(1), tau_fit, y_max)
    line_tau.SetLineStyle(2); line_tau.SetLineColor(ROOT.kRed); line_tau.SetLineWidth(2)
    line_tau.Draw()
    fit_funcs_global.append((f, line_tau))

c_final.Draw()

## Cell 11 — Source activities
$r_i = \lambda_i - \lambda_{i-1}$  (activity of the $i$-th source added).  
The uncertainty on each difference propagates from the fit uncertainties on $\lambda_i$.

In [ ]:
# --- Covariance of λᵢ from the Hessian of the global NLL ---
# The analytic variance of λ̂ᵢ for fixed τ is:
#   Var(λᵢ) = λᵢ² / nᵢ     (Cramér-Rao bound for shifted exponential)
# This is a fast approximation; for the exact result use the Hessian.
lam_vars = lams_fit**2 / ns          # variance of each λᵢ
lam_errs = np.sqrt(lam_vars)

# Activities
activities     = np.zeros(N_RUNS)
activity_errs  = np.zeros(N_RUNS)

activities[0]    = lams_fit[0]
activity_errs[0] = lam_errs[0]

for i in range(1, N_RUNS):
    activities[i]   = lams_fit[i] - lams_fit[i-1]
    activity_errs[i] = np.sqrt(lam_vars[i] + lam_vars[i-1])

print('=' * 62)
print(f'  Dead time  τ = {tau_fit:.6e} s'
      f'   [{tau_1sig_lo:.3e}, {tau_1sig_hi:.3e}]  (1σ)')
print('=' * 62)
print(f"{'Source':>8}  {'λᵢ (Hz)':>12}  {'δλᵢ':>10}  {'rᵢ (Hz)':>12}  {'δrᵢ':>10}")
print('-' * 62)
for i in range(N_RUNS):
    print(f"{i+1:>8}  {lams_fit[i]:>12.4f}  {lam_errs[i]:>10.4f}"
          f"  {activities[i]:>12.4f}  {activity_errs[i]:>10.4f}")
print('=' * 62)

# Sanity check: all activities should be positive
if np.all(activities > 0):
    print('\n✓ All individual source activities are positive — consistent fit.')
else:
    print('\n⚠ Some activities are negative — check data or fitting range.')

## Cell 12 — Summary bar chart of activities

In [ ]:
c_bar = ROOT.TCanvas('c_bar', 'Source activities', 800, 500)

g_act = ROOT.TGraphErrors(
    N_RUNS,
    np.arange(1, N_RUNS+1, dtype='d'),
    activities.astype('d'),
    np.zeros(N_RUNS, dtype='d'),
    activity_errs.astype('d')
)
g_act.SetTitle('Individual source activities;Source index;Activity (Hz)')
g_act.SetMarkerStyle(20)
g_act.SetMarkerSize(1.5)
g_act.SetMarkerColor(ROOT.kBlue+1)
g_act.SetLineColor(ROOT.kBlue+1)
g_act.SetLineWidth(2)
g_act.Draw('AP')
c_bar.Draw()